In [1]:
import sys 
!"{sys.executable}" -m pip install networkx numpy scipy torch scikit-learn matplotlib
# torch-geometric may require special install steps on Windows and is optional for this notebook



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load C_elegans edgelist as an unweighted graph
path = os.path.join("..","Datasets", "C_elegans.txt")

# Each line: V1 V2 weight. We ignore the weight and use binary edges.
edges = np.loadtxt(path, dtype=int, usecols=(0, 1))

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")

Nodes: 297
Edges: 2148
Graph is large; skipping full plot.


In [4]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [5]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([ 11,  29,  74,  52,  54,  14,  36,  18,  20,  17,  13,  11,  77,
        32,  24,  15,  19,  10,  16,  11,  19,  21,  15,  20,   7,   6,
         7,  19,   6,  10,  11,  21,  11,  14,  12,  17,   9,   6,   9,
        10,   8,  14,  24,  17, 134,   8,  16,  20,  23,  14,  17,  10,
        17,  11,   8,   8,   8,   9,  23,  32,   8,   9,  12,  26,  13,
        12,  11,  21,   9,  12,  10,  21,  16,  18,  19,  15,   8,  15,
         7,   5,   2,   3,   9,   9,  53,  15,  53,  21,   6,  12,  26,
        13,   5,  22,   4,  16,   7,  17,  30,  10,   9,  17,  20,  21,
        11,  17,  14,  11,  20,  17,  10,  22,  11,  16,  33,  15,  15,
        49,  53,  29,  10,  16,  15,  21,  12,  51,   4,   9,   4,  14,
        29,  12,  25,  16,   8,  15,   8,  46,  12,  11,   6,  10,  37,
         5,   2,  22,   5,   6,  18,  17,  13,  15,  11,  12,  10,  11,
        10,  13,   9,  12,  15,  13,  14,  15,  10,  10,  15,  14,  16,
        15,  12,  11,  52,  12,   6,   8,  11,  22,  15,  14,  1

In [6]:
dist = dict(nx.all_pairs_shortest_path_length(G))
dist

{np.int64(1): {np.int64(1): 0,
  np.int64(51): 1,
  np.int64(72): 1,
  np.int64(77): 1,
  np.int64(78): 1,
  np.int64(2): 1,
  np.int64(90): 1,
  np.int64(92): 1,
  np.int64(158): 1,
  np.int64(159): 1,
  np.int64(53): 1,
  np.int64(145): 1,
  np.int64(47): 2,
  np.int64(113): 2,
  np.int64(58): 2,
  np.int64(71): 2,
  np.int64(73): 2,
  np.int64(154): 2,
  np.int64(96): 2,
  np.int64(127): 2,
  np.int64(128): 2,
  np.int64(140): 2,
  np.int64(54): 2,
  np.int64(55): 2,
  np.int64(56): 2,
  np.int64(57): 2,
  np.int64(59): 2,
  np.int64(61): 2,
  np.int64(63): 2,
  np.int64(67): 2,
  np.int64(87): 2,
  np.int64(88): 2,
  np.int64(109): 2,
  np.int64(123): 2,
  np.int64(141): 2,
  np.int64(178): 2,
  np.int64(185): 2,
  np.int64(199): 2,
  np.int64(200): 2,
  np.int64(48): 2,
  np.int64(52): 2,
  np.int64(62): 2,
  np.int64(70): 2,
  np.int64(144): 2,
  np.int64(146): 2,
  np.int64(225): 2,
  np.int64(186): 2,
  np.int64(226): 2,
  np.int64(227): 2,
  np.int64(228): 2,
  np.int64(229): 

In [7]:
n = len(nodelist)
dist_matrix = np.zeros((n, n))

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and v in dist[u]:
            dist_matrix[i, j] = dist[u][v]

print(dist_matrix)

[[0. 1. 1. ... 4. 4. 4.]
 [1. 0. 2. ... 3. 3. 3.]
 [1. 2. 0. ... 3. 3. 3.]
 ...
 [4. 3. 3. ... 0. 4. 4.]
 [4. 3. 3. ... 4. 0. 2.]
 [4. 3. 3. ... 4. 2. 0.]]


4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [8]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [497.13293423 547.54830892 667.07465072 630.35800129 634.24319054
 506.43125736 550.68301261 526.54819512 533.65159075 503.77250067
 462.2770402  504.18917261 673.60090497 545.40490959 556.64799651
 485.71169307 532.89805839 492.21891249 489.69136094 445.10337414
 525.43375496 524.83477823 462.52410653 533.20383803 494.159406
 436.79175655 465.51642097 494.82687518 410.10893758 496.5199974
 439.36471844 506.37401335 436.92846331 455.32917919 517.64731598
 527.24190904 502.39342235 422.46140161 474.79482409 449.47824746
 484.73280601 516.66248084 540.80902728 528.24048508 741.03826577
 489.03876654 524.53127145 548.17259086 562.01248589 493.33772306
 529.21469823 474.18987041 510.49007064 491.23043488 412.86982926
 430.22211022 478.98570669 424.57713662 504.94975012 527.8498883
 496.10720802 484.34372828 472.51110992 545.59199196 509.37058382
 496.27347814 442.9257104  543.56879125 435.61086599 473.84744938
 433.25273835 504.22870155 524.49456128 552.49658309 549.33324048
 538.1680

### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [9]:
NLI = np.zeros(n)

for i, u in enumerate(nodelist):

    neighbors = list(G.neighbors(u))

    # Sum of exponential terms (using neighbor count here as influence proxy)
    exp_sum = 0
    for v in neighbors:
        exp_sum += np.exp(G.degree(v))   # you can modify this part if Ne_i(v_i) defined differently

    if exp_sum > 0:
        NLI[i] = (G.degree(u) * np.log2(exp_sum)) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

#calculation Verified

NLI: [ 3.95405308 10.84692938 27.67837153 19.44966648 19.41080601  5.23644867
  8.04411809  6.47026867  7.18918741  4.45923922  2.33693115  4.11435252
 27.67837153  7.15033447  8.97676914  3.93462284  7.11109321  6.50912914
  4.19693103  1.70986104  7.11109321  7.85962933  2.3316542  13.01825828
  4.5563904   1.0494285   4.5563904   4.79926836  0.96179672  6.50912914
  1.9245632   5.52141974  1.70986184  2.1762096   7.81095497 11.06551954
  5.85821623  0.90350979  5.85821623  1.55442127  5.20730331  9.1127808
  6.28582987 11.06551954 24.08574362  5.20730331 10.41460663 13.01825828
 14.97099702  9.1127808  11.06551954  6.50912914  4.5047767   2.90212864
  0.93265803  1.24353643  5.20730331  1.39898314  5.80964064  5.59874458
  5.20730331  5.85821623  3.03111685  9.73096965  8.46186788  3.10766781
  1.70986292  7.85962933  1.3989835   3.08940756  1.58808893  5.50847197
  5.75134993 11.71643245 12.36734537  9.76369371  2.09846551  9.76369371
  1.73414858  1.26296536  0.30116866  0.4518508

4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [10]:
NLGC = NLI * NGI
NLGC

array([ 1965.69000734,  5939.21783985, 18463.54001762, 12260.25288637,
       12311.1715317 ,  2651.90128251,  4429.75918614,  3406.90828922,
        3836.52129723,  2246.44209158,  1080.30961683,  2074.41199491,
       18644.17610794,  3899.82752761,  4996.90055873,  1911.09232064,
        3789.48776348,  3203.91646692,  2055.20086691,   761.06491881,
        3736.40840601,  4125.00681896,  1078.44627437,  6941.38528014,
        2251.58317282,   458.3817189 ,  2121.07455085,  2374.80696678,
         394.44143046,  3231.91278401,   845.58516768,  2795.90347356,
         747.08730483,   990.89172994,  4043.3198749 ,  5834.20564639,
        2943.12929895,   381.69801032,  2781.45074282,   698.67854833,
        2524.15074645,  4708.23193393,  3399.43353901,  5845.25540901,
       17848.45768559,  2546.57318899,  5462.78685478,  7136.25237057,
        8413.88725349,  4495.67852915,  5856.03558371,  3086.56310371,
        2299.64377454,  1425.61391413,   385.06636255,   534.9968675 ,
      

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [11]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [ 3.95405308 10.84692938 27.67837153 19.44966648 19.41080601  5.23644867
  8.04411809  6.47026867  7.18918741  4.45923922  2.33693115  4.11435252
 27.67837153  7.15033447  8.97676914  3.93462284  7.11109321  6.50912914
  4.19693103  1.70986104  7.11109321  7.85962933  2.3316542  13.01825828
  4.5563904   1.0494285   4.5563904   4.79926836  0.96179672  6.50912914
  1.9245632   5.52141974  1.70986184  2.1762096   7.81095497 11.06551954
  5.85821623  0.90350979  5.85821623  1.55442127  5.20730331  9.1127808
  6.28582987 11.06551954 24.08574362  5.20730331 10.41460663 13.01825828
 14.97099702  9.1127808  11.06551954  6.50912914  4.5047767   2.90212864
  0.93265803  1.24353643  5.20730331  1.39898314  5.80964064  5.59874458
  5.20730331  5.85821623  3.03111685  9.73096965  8.46186788  3.10766781
  1.70986292  7.85962933  1.3989835   3.08940756  1.58808893  5.50847197
  5.75134993 11.71643245 12.36734537  9.76369371  2.09846551  9.76369371
  1.73414858  1.26296536  0.30116866  0.4518

### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [12]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Nodes used: [np.int64(1), np.int64(72), np.int64(78), np.int64(77), np.int64(145), np.int64(90), np.int64(51), np.int64(158), np.int64(92), np.int64(159), np.int64(2), np.int64(53), None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [13]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [14]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[ 3.93462284 19.41080601  6.84146388 ...  0.          0.
   0.        ]
 [ 3.93462284 19.41080601  6.84146388 ...  0.          0.
   0.        ]
 [ 3.93462284 19.41080601  6.84146388 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]
E_NLI2:
 [[ 98.9214241  504.86404929 275.2118813  ...   0.           0.
    0.        ]
 [ 98.9214241  504.86404929 275.2118813  ...   0.           0.
    0.        ]
 [ 98.9214241  504.86404929 275.2118813  ...   0.           0.
    0.        ]
 ...
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]]
E_NLI3:
 [[ 2424.30805215 13413.51121488  6108.49432032 ..

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [15]:
channels = []

for node in G.nodes():

    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1)
    tensor = np.stack([c1, c2, c3, c4, c5, c6])

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1)
channels = np.array(channels)

print(channels.shape)

(297, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [16]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [17]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [19]:
import numpy as np
import random

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)

beta = 1.5 * beta_c
mu = 1


# ---- SIR Simulation ----
def SIR_simulation(G, seed, beta, mu, steps=1000):

    susceptible = set(G.nodes())
    infected = {seed}
    recovered = set()

    susceptible.remove(seed)

    for _ in range(steps):

        new_infected = set()
        new_recovered = set()

        for node in infected:

            # spread infection
            for nbr in G.neighbors(node):
                if nbr in susceptible:
                    if random.random() < beta:
                        new_infected.add(nbr)

            # recovery
            if random.random() < mu:
                new_recovered.add(node)

        infected |= new_infected
        infected -= new_recovered

        recovered |= new_recovered
        susceptible -= new_infected

        if len(infected) == 0:
            break

    return len(recovered)


# ---- Label Generation ----
labels = []
runs = 500

for node in G.nodes():

    spread = 0

    for _ in range(runs):
        spread += SIR_simulation(G, node, beta, mu)

    labels.append(spread / runs)

labels = np.array(labels)


# ---- Normalize Labels ----
labels = labels / np.max(labels)

print("Labels:", labels)

Labels: [0.32117208 0.52229963 0.99773772 0.86411951 0.86964953 0.37819592
 0.57020253 0.43349612 0.50369865 0.39378052 0.23222494 0.33966533
 1.         0.56266159 0.52517236 0.3088193  0.43600977 0.26396869
 0.34839127 0.16733697 0.41241741 0.42092789 0.2570023  0.38968687
 0.19556162 0.15484056 0.14647371 0.35550129 0.19046251 0.21491669
 0.23538495 0.39159006 0.17175381 0.21929762 0.27890692 0.34006033
 0.19365843 0.16033467 0.16360241 0.20400029 0.1615915  0.29894427
 0.53651968 0.35837403 0.98563631 0.21735852 0.32013071 0.46696352
 0.4976659  0.24490089 0.33952169 0.224289   0.38128411 0.30476156
 0.16026286 0.1300632  0.14981327 0.13462367 0.43507613 0.47436082
 0.20123528 0.22511491 0.26350187 0.52466964 0.29036196 0.31021976
 0.23904769 0.55260701 0.18446567 0.25262137 0.19882936 0.37629273
 0.42114335 0.43992387 0.39209279 0.35054582 0.23836541 0.37841138
 0.19294025 0.13329503 0.06794025 0.08115484 0.23150675 0.23814996
 0.91029876 0.36131859 0.96997989 0.47529446 0.1527219

In [20]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([ 6.4060e-08, -4.5233e-09,  2.2739e-08,  7.7386e-08, -1.9438e-08,
         1.9499e-07])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.33623966574668884 y std: 0.18955206871032715
Epoch 0, Loss = 1.428473
Epoch 20, Loss = 0.229314
Epoch 40, Loss = 0.197736
Epoch 60, Loss = 0.182826
Epoch 80, Loss = 0.171704
Epoch 100, Loss = 0.161959
Epoch 120, Loss = 0.153210
Epoch 140, Loss = 0.145131
Epoch 160, Loss = 0.137610
Epoch 180, Loss = 0.130565
Epoch 200, Loss = 0.123999
Epoch 220, Loss = 0.117883
Epoch 240, Loss = 0.112166
Epoch 260, Loss = 0.106861
Epoch 280, Loss = 0.101818

Final Predictions:
 tensor([[-0.0636],
        [ 0.9832],
        [ 3.4931],
        [ 2.7903],
        [ 2.8182],
        [ 0.1794],
        [ 1.2370],
        [ 0.4751],
        [ 0.8740],
        [ 0.3656],
        [-0.5056],
        [-0.0131],
        [ 3.5067],
        [ 1.1978],
        [ 1.0191],
        [-0.2039],
        [ 0.5956],
        [-0

### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [21]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [-0.06357777  0.9831616   3.493114    2.7902815   2.818166    0.17938244
  1.2370381   0.47507733  0.87395483  0.36562747 -0.5055837  -0.01311886
  3.5066767   1.1978374   1.0190787  -0.20393655  0.59559506 -0.5252462
  0.05376107 -0.5252462   0.4025725   0.42922503 -0.38736838  0.24604827
 -0.5252462  -0.5252462  -0.5252462   0.13073325 -0.5252462  -0.5252462
 -0.5252462   0.31773216 -0.5252462  -0.5157781  -0.25893527  0.03084224
 -0.52018535 -0.5252462  -0.5252462  -0.5252462  -0.5252462  -0.21701539
  1.0529375   0.11936873  3.4350667  -0.5252462  -0.07866603  0.6806982
  0.8514318  -0.46012944  0.06981027 -0.5252462   0.2105906  -0.2479955
 -0.5252462  -0.5252462  -0.5252462  -0.5252462   0.5239174   0.73340124
 -0.5252462  -0.5252462  -0.2930342   0.9955208  -0.28165406 -0.09211621
 -0.5252462   1.1219511  -0.5252462  -0.3978001  -0.5252462   0.2048065
  0.49160784  0.5555845   0.32010573  0.0749082  -0.5252462   0.12158728
 -0.5252462  -0.5252462  -0.5252462  -

###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [22]:
from scipy.stats import kendalltau

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

spread_total = 0

for node in top_nodes:
    spread_total += SIR_simulation(G, node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)

Kendall Tau: 0.9106601869543066
Top-N node indices: [12  2 44]
Top-N nodes: [np.int64(71), np.int64(72), np.int64(305)]
Spread ability: 156


In [23]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for c elegans...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for c elegans...
Kendall Tau (Prediction vs Degree): 0.7308
Kendall Tau (Prediction vs Betweenness): 0.5555
Kendall Tau (Prediction vs Closeness): 0.6504
Kendall Tau (Prediction vs PageRank): 0.6633
Kendall Tau (Prediction vs Coreness): 0.7394
Kendall Tau (Prediction vs Eigenvector): 0.8170
